In [37]:
import src.db_read as dbrd
import src.save_grid as svgrd
import src.grid_topol as grdtpl
import src.weather as wth 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import pandapower as pp
from pathlib import Path

import contextily as ctx
from shapely.geometry import box
from shapely.geometry import Point
import geopandas as gpd
from pyproj import Transformer



### Load census and grids

In [38]:
lat, lon = 52.5200, 13.4050
match_by = 'distance'

In [39]:
df_grid_set = pd.read_hdf("input_data/valid_grids")  # Source: pylovo
df_grid_set.head()

,plz,kcid,bcid,x,y,Einwohner,x_census,y_census
0,90461,1,22,4.400701e+06,2.925878e+06,13424.0,4400500.0,2925500.0
1,80933,1,-2,4.437005e+06,2.790850e+06,2579.0,4437500.0,2790500.0
2,81829,1,21,4.446704e+06,2.780981e+06,1255.0,4446500.0,2780500.0
3,81829,1,22,4.446772e+06,2.781089e+06,4195.0,4446500.0,2781500.0
4,81927,1,22,4.442523e+06,2.783343e+06,5266.0,4442500.0,2783500.0


In [40]:
df_census = pd.read_csv("input_data/Zensus2022_Bevoelkerungszahl_1km-Gitter.csv", sep=";")  # Source: https://atlas.zensus2022.de/
df_census.head()

,GITTER_ID_1km,x_mp_1km,y_mp_1km,Einwohner
0,CRS3035RES1000mN2689000E4337000,4337500,2689500,4
1,CRS3035RES1000mN2689000E4341000,4341500,2689500,11
2,CRS3035RES1000mN2690000E4341000,4341500,2690500,4
3,CRS3035RES1000mN2691000E4340000,4340500,2691500,3
4,CRS3035RES1000mN2691000E4341000,4341500,2691500,22


### Sample a set of representative grids (N~1500)

1. Sample census cell based on number of inhabitants (p(cell)~N_inhabitants)
2. If cell contains 1 or more simulated grids:

    a) Sample one of the contained grids randomly (all with equal probability)

    b) Assign as grid position the transformer position


3. If cell contains no simulated grid:
    
    a) Take all grids with the closest population density to the census cell

    b) Sample one grid out of this subset

    c) Take as grid position the center of the census cell

4. From position: assign zip code and regiostar region

##### 1. Sample 1500 census cells

In [41]:
def find_nearest_census_cell(lat, lon, df_census):
    """
    Find the nearest census cell to the given lat/lon coordinates.

    Args:
        lat: Latitude (EPSG:4326)
        lon: Longitude (EPSG:4326)
        df_census: DataFrame with census data (columns: x_mp_1km, y_mp_1km, Einwohner)

    Returns:
        Series with the nearest census cell data
    """
    # Transform from WGS84 (EPSG:4326) to EPSG:3035
    transformer = Transformer.from_crs("EPSG:4326", "EPSG:3035", always_xy=True)
    x_census, y_census = transformer.transform(lon, lat)

    print(f"Input coordinates: lat={lat}, lon={lon}")
    print(f"Transformed to EPSG:3035: x={x_census:.2f}, y={y_census:.2f}")

    # Find the nearest census cell by Euclidean distance
    distances = (df_census["x_mp_1km"] - x_census) ** 2 + (
        df_census["y_mp_1km"] - y_census
    ) ** 2
    nearest_idx = distances.idxmin()
    nearest_cell = df_census.loc[nearest_idx]

    print(
        f"Nearest census cell: x={nearest_cell['x_mp_1km']}, y={nearest_cell['y_mp_1km']}, "
        f"population={nearest_cell['Einwohner']}"
    )

    return nearest_cell

In [42]:
# Normalize inhabitants to get probabilities
probabilities = df_census['Einwohner'] / df_census['Einwohner'].sum()

# Sample 1000 rows with replacement=False, weighted by inhabitants
df_sampled_cells = pd.DataFrame(find_nearest_census_cell(lat, lon, df_census)).T  # Example for Berlin
# df_sampled_cells = df_census.sample(n=1500, weights=probabilities, replace=True, random_state=3)
df_sampled_cells.head()

Input coordinates: lat=52.52, lon=13.405
Transformed to EPSG:3035: x=4552036.45, y=3273268.27
Nearest census cell: x=4552500, y=3273500, population=8544


,GITTER_ID_1km,x_mp_1km,y_mp_1km,Einwohner
159779,CRS3035RES1000mN3273000E4552000,4552500,3273500,8544


In [43]:
df_sampled_cells.duplicated().sum()

np.int64(0)

In [44]:
# Mean of sample matches expectation value, so all good
(df_census["Einwohner"]**2).sum()/df_census["Einwohner"].sum()

np.float64(3082.4239705301843)

In [45]:
# Step 1: Column definitions
df=df_sampled_cells.copy()
x_col    = "x_mp_1km"    # centroid X [m]
y_col    = "y_mp_1km"    # centroid Y [m]
pop_col  = "Einwohner"    # population

half_size = 500          # half side‐length in metres

# Step 2: Build square polygons in EPSG:3035
polygons = df.apply(
    lambda row: box(
        row[x_col] - half_size,
        row[y_col] - half_size,
        row[x_col] + half_size,
        row[y_col] + half_size
    ),
    axis=1
)

gdf = gpd.GeoDataFrame(
    df[[pop_col]],
    geometry=polygons,
    crs="EPSG:3035"
)

# Step 3: Plot in EPSG:3035 so boxes remain true squares
# fig, ax = plt.subplots(figsize=(10, 10))

# Plot the grid cells
# gdf.plot(
#     ax=ax,
#     column=pop_col,
#     cmap="viridis",
#     linewidth=0,
#     edgecolor="none",
#     alpha=0.8,
#     legend=True,
#     legend_kwds={
#         "label": "Population per 1 km² grid cell",
#         "shrink": 0.6
#     }
# )

# Optional: add a basemap reprojected on the fly
# ctx.add_basemap(
#     ax,
#     source=ctx.providers.CartoDB.Positron,
#     crs=gdf.crs.to_string()
# )

# # Formatting
# ax.set_title(
#     "Deutschland – Bevölkerung pro 1 km × 1 km (Zensus 2022), EPSG:3035",
#     fontsize=14
# )
# ax.set_aspect("equal")      # ensure equal axis scales
# ax.set_axis_off()

# plt.tight_layout()
# plt.show()

##### 2. + 3. Assign Representative Grid To Cell

In [46]:
### Step 2 + 3 - Assign grid to cell: ###
def assign_grid(x_cell, y_cell, N_inh, df_grids, match_by="population"):
    df_g = df_grids.copy()

    ### 2a) Check if cell contains sampled grids already:
    df_same_cell = df_g[(df_g["x_census"]==x_cell) & (df_g["y_census"]==y_cell)]
    if len(df_same_cell)!=0:
        sample_grid = df_same_cell.sample(n=1, random_state=x_cell)

        #2b) Assign trafo pos as grid pos
        sample_grid.drop(columns=["x_census", "y_census"], inplace=True)
        return sample_grid
    
    
    ### 3a) Find grids based on matching strategy:
    if match_by == "population":
        # Match by closest population density
        df_g["diff"] = (df_g["Einwohner"] - N_inh).abs()
        min_diff = df_g["diff"].min()
        closest_rows = df_g[df_g["diff"] == min_diff].drop(columns="diff")

        print(
            f"No grid in census cell. Found {len(closest_rows)} grid(s) with closest population density "
            f"(diff={min_diff:.1f}), using cell center as position"
        )

    elif match_by == "distance":
        # Match by geographic distance to census cell center
        df_g["dist"] = (
            (df_g["x_census"] - x_cell) ** 2 + (df_g["y_census"] - y_cell) ** 2
        ) ** 0.5
        min_dist = df_g["dist"].min()
        closest_rows = df_g[df_g["dist"] == min_dist].drop(columns="dist")

        print(
            f"No grid in census cell. Found {len(closest_rows)} grid(s) at closest distance "
            f"({min_dist:.1f}m from cell center), using cell center as position"
        )

    else:
        raise ValueError(
            f"Invalid match_by value: {match_by}. Must be 'population' or 'distance'"
        )

    # 3b) Sample one grid out of selection
    sample_grid = closest_rows.sample(n=1, random_state=x_cell)

    # 3c) Take cell center as trafo pos
    sample_grid[["x","y"]] = (x_cell, y_cell)
    sample_grid.drop(columns=["x_census", "y_census"], inplace=True)
    return sample_grid

In [47]:
df_sampled_grids = df_sampled_cells.apply(lambda row: assign_grid(row["x_mp_1km"], row["y_mp_1km"], row["Einwohner"], df_grid_set, match_by), axis=1)
df_sampled_grids = pd.concat(df_sampled_grids.values, ignore_index=True).rename(columns={"plz":"plz_pylovo", "Einwohner":"pop_density_1km2_cell"})
df_sampled_grids.head()

No grid in census cell. Found 2 grid(s) at closest distance (278729.3m from cell center), using cell center as position


,plz_pylovo,kcid,bcid,x,y,pop_density_1km2_cell
0,95213,2,10,4552500,3273500,24.0


In [48]:
df_sampled_grids.duplicated().sum()

np.int64(0)

In [49]:
df_sampled_grids["pop_density_1km2_cell"].describe()

count     1.0
mean     24.0
std       NaN
min      24.0
25%      24.0
50%      24.0
75%      24.0
max      24.0
Name: pop_density_1km2_cell, dtype: float64

#### 4. Assign zip-code and regiostar7

Lattitude and Longitude

In [50]:
transformer = Transformer.from_crs("EPSG:3035", "EPSG:4326", always_xy=True)
df_sampled_grids[["lon","lat"]] = df_sampled_grids.apply(lambda row: transformer.transform(row["x"], row["y"]), axis=1).apply(pd.Series)
df_sampled_grids.head()

,plz_pylovo,kcid,bcid,x,y,pop_density_1km2_cell,lon,lat
0,95213,2,10,4552500,3273500,24.0,13.411983,52.521885


zip code

In [51]:
shapefile_path = "input_data/plz-5stellig.shp"  # Source: https://www.suche-postleitzahl.org/downloads
gdf_zip = gpd.read_file(shapefile_path)

In [52]:
def lookup_zip_code(lon, lat, gdf_zip):
    point = Point(lon, lat)  # Note: shapely expects (x, y) = (lon, lat)

    # Ensure the CRS is set and transform if necessary
    if gdf_zip.crs is None:
        gdf_zip.set_crs(epsg=4326, inplace=True)
    elif gdf_zip.crs.to_epsg() != 4326:
        gdf_zip = gdf_zip.to_crs(epsg=4326)

    # Spatial join to find the matching PLZ polygon
    matching_row = gdf_zip[gdf_zip.contains(point)]

    # Extract PLZ
    if not matching_row.empty: return matching_row.iloc[0]["plz"]
    else: return np.nan

In [53]:
df_sampled_grids["plz"] = df_sampled_grids.apply(lambda row: lookup_zip_code(row["lon"], row["lat"], gdf_zip), axis=1)
df_sampled_grids = df_sampled_grids.dropna(subset=['plz']) # Drop those for which Gemeindeschlüssel could not be found (shouldn't be more than 1%)
df_sampled_grids.head()

,plz_pylovo,kcid,bcid,x,y,pop_density_1km2_cell,lon,lat,plz
0,95213,2,10,4552500,3273500,24.0,13.411983,52.521885,10178


Gemeindeschlüssel

In [54]:
# Path to your downloaded VG250 shapefile:
MUNICI_SHP = 'input_data/VG250_GEM.shp'     # Source: https://gdz.bkg.bund.de/index.php/default/verwaltungsgebiete-1-250-000-stand-01-01-vg250-01-01.html

# Read municipalities
gdf_munic = gpd.read_file(MUNICI_SHP)

# Make sure it’s in WGS84 (lat/lon)
if gdf_munic.crs.to_epsg() != 4326:
    gdf_munic = gdf_munic.to_crs(epsg=4326)

# Build spatial index
sindex = gdf_munic.sindex

In [55]:
def lookup_gemeindeschluessel(lat, lon):
    """
    Given WGS84 latitude and longitude,
    returns the matching Gemeinde-Schlüssel (AGS) and name.
    """
    pt = Point(lon, lat)

    # first find candidate polygons
    idx_candidates = list(sindex.intersection(pt.bounds))
    candidates = gdf_munic.iloc[idx_candidates]

    # then test which one contains the point
    match = candidates[candidates.contains(pt)]
    if match.empty:
        return None

    row = match.iloc[0]
    return row['AGS'], row['GEN']   # Gemeindeschlüssel + municipality name

In [56]:
df_sampled_grids[["gemeindeschlüssel", "name"]] = df_sampled_grids.apply(lambda row: lookup_gemeindeschluessel(row["lat"], row["lon"]), axis=1).apply(pd.Series)
df_sampled_grids = df_sampled_grids.dropna(subset=['gemeindeschlüssel']) # Drop those for which Gemeindeschlüssel could not be found (shouldn't be more than 1%)
df_sampled_grids.head()

,plz_pylovo,kcid,bcid,x,y,pop_density_1km2_cell,lon,lat,plz,gemeindeschlüssel,name
0,95213,2,10,4552500,3273500,24.0,13.411983,52.521885,10178,11000000,Berlin


RegioStar7

In [57]:
df_regiostar = pd.read_excel("input_data/regiostar-referenzdateien.xlsx", sheet_name="ReferenzGebietsstand2020")
df_regiostar.head()

,gem_20,gemrs_20,name_20,bev_20,fl_20,vbgem_20,vbgemrs_20,vbgnam_20,land_20,RegioStaR2,RegioStaR4,RegioStaR17,RegioStaR7,RegioStaR5,RegioStaRGem7,RegioStaRGem5,RegioStaR_Stadtregion,RegioStaR_NameStadtregion
0,1001000,10010000000,"Flensburg, Stadt",89934,56.73,1001000,10010000,"Flensburg, Stadt",1,2,22,221,75,54,74,53,NaN,NaN
1,1002000,10020000000,"Kiel, Landeshauptstadt",246601,118.65,1002000,10020000,"Kiel, Landeshauptstadt",1,1,12,121,72,52,72,52,1002000.0,Kiel
2,1003000,10030000000,"Lübeck, Hansestadt",215846,214.19,1003000,10030000,"Lübeck, Hansestadt",1,1,12,121,72,52,72,52,1003000.0,Lübeck
3,1004000,10040000000,"Neumünster, Stadt",79905,71.66,1004000,10040000,"Neumünster, Stadt",1,2,21,211,75,54,74,53,NaN,NaN
4,1051001,10515175001,Albersdorf,3714,17.12,1051975,10515175,Mitteldithmarschen,1,2,22,225,77,55,77,55,NaN,NaN


In [58]:
def get_regiostar_region(AGS, df_regiostar):
    """Get RegioStaR7 typology from Gemeindeschlüssel (AGS)"""
    row = df_regiostar[df_regiostar["gem_20"]==AGS]
    try: return int(row["RegioStaR7"].values[0])
    except: return np.nan

In [59]:
df_sampled_grids["regio7"] = df_sampled_grids.apply(lambda row: get_regiostar_region(int(row["gemeindeschlüssel"]), df_regiostar), axis=1)
df_sampled_grids = df_sampled_grids.dropna(subset=['regio7'])  # Drop those for which regiostar7 could not be found, shouldn't be more than 1%

In [60]:
# Drop general duplicate entries (this will slightly skew the distribution, but prevents simulating the same grid several times):
df_sampled_grids = df_sampled_grids.drop_duplicates().reset_index(drop=True)
df_sampled_grids

,plz_pylovo,kcid,bcid,x,y,pop_density_1km2_cell,lon,lat,plz,gemeindeschlüssel,name,regio7
0,95213,2,10,4552500,3273500,24.0,13.411983,52.521885,10178,11000000,Berlin,71


### Read out pylovo grid data

In [61]:
### Create database object to read grid data from pylovo database
DB = dbrd.DataBase()
DB.show_contents()

Available tables:
['spatial_ref_sys', 'geography_columns', 'geometry_columns', 'res', 'oth', 'betriebsmittel', 'building_clusters', 'lines_result', 'buildings_result', 'sample_set', 'clustering_parameters', 'classification_version', 'buildings_tem', 'consumer_categories', 'postcode_result', 'transformer_classified', 'ags_log', 'ways', 'loadarea', 'postcode', 'transformer_positions', 'transformers', 'ways_result', 'ways_tem', 'grids', 'version', 'grid_parameters', 'ways_tem_vertices_pgr', 'public_2po_4pgr', 'municipal_register']


In [62]:
def retrieve_pylovo_grid(df_region_specs):
    ''' :: df_region_specs must at least include columns:
        - plz_pylovo
        - kcid, bcid
        - plz
        - regio7
        - lat, lon
    '''

    grid_specs = {    # unique grid identifier (as duplicates were dropped)
        "cell_id": f"N{int(df_region_specs['y'])}E{int(df_region_specs['x'])}",
        "plz":  df_region_specs["plz_pylovo"],
        "kcid": df_region_specs["kcid"],
        "bcid": df_region_specs["bcid"]
    }
    # Generate SaveFile object to find file path and test if file exists
    # SF=svgrd.SaveFile(grid_specs)
    # Path("/data/input/", f"grid_{SF.path.split('/')[-1].split('.')[0]}.xlsx")

    # if Path.exists(Path("/data/input/", f"grid_{SF.path.split('/')[-1].split('.')[0]}.xlsx")):
    #     print(f"Grid file already exists at grid_{SF.path.split('/')[-1].split('.')[0]}.xlsx, skipping retrieval.")
    #     return(SF.path)

    ######### Process Pylovo Grid #########
    # Read out pandapower grid associated with plz, kcid, bcid
    net = DB.read_single_ppgrid(grid_specs)
    net = grdtpl.assign_min_linelen(net)        # Adjust and save grid topology
    net = grdtpl.remove_duplicate_loads(net)

    # Retrieve buildings associated with plz, kcid, bcid
    df_buildings = DB.read_buildings(grid_specs, net.bus)

    ######## Process Weather ###########
    location = {"lat": df_region_specs["lat"], 
                "lon": df_region_specs["lon"]
            }

    # Get TMY data from SARAH3 dataset as DataFrame
    df_weather_raw, altitude, selected_months = wth.get_pvgis_tmy_sarah3_dataframe(location["lat"], location["lon"])
    # Add dew point temperature necessary for vehicle simulation
    df_weather_raw["dew_point"] = wth.get_dew_point(df_weather_raw["temp_air"], df_weather_raw["relative_humidity"])
    # Add soil temperature (1.00-2.55m) necessary for ground source heat pumps
    df_weather_raw["soil_temp"] = wth.get_open_meteo_soil_temperature(location["lat"], location["lon"], selected_months)

    df_region_specs["altitude"] = altitude

    ######### Save to file #########
    SF=svgrd.SaveFile(grid_specs)
    SF.save_topology(net, "/raw_data/")
    SF.save_df(df_region_specs, "/raw_data/region")
    SF.save_df(df_buildings, "/raw_data/buildings")
    SF.save_df(df_weather_raw, "/raw_data/weather")
    pp.to_excel(net, f"/data/input/grid_{SF.path.split('/')[-1].split('.')[0]}.xlsx")

    return SF.path, net

In [63]:
SF_paths, net = df_sampled_grids[0:100].apply(retrieve_pylovo_grid, axis=1)[0]

Requesting TMY data from PVGIS (SARAH3) for coordinates (52.521884734529415, 13.411983346090851)...
File data/grids/N3273500E4552500_95213_2_10.h5 created!


# Demand Generation


In [64]:
import src.classes.grid as grd

In [65]:
### Run Settings
settings = {
    "grid_filename": SF_paths.split("/")[-1],     # Name of input file
    # "grid_filename" = "N2827500E4503500_93426_5_41.h5",     
    "weather_data_exists": True,                           # Is weather data already included in input grid file's raw data? (recommended, as on HPC cluster no outside API access)
    "parallel": True,                                      # Parallelized run?
    "n_cpu": 12                                            # cpus if parallel 
}                    

# Setup grid which stores all relevant data for assigning demands
GRD = grd.Grid(settings)
# GRD.df_buildings = GRD.df_buildings.iloc[0:10].reset_index(drop=True)

In [66]:
GRD.df_buildings

,bus,osm_id,vertice_id,type,use,houses_per_building,occupants,free_walls,floors,constructi,area,lat,lon
0,14,445138602,3633,SFH,Residential,2,4.0,4.0,1,2009-,174.339721,50.226958,11.805801
1,15,377332551,3639,SFH,Residential,1,0.0,4.0,1,1991-1995,114.292871,50.230866,11.804509
2,16,155903803,3642,SFH,Residential,2,0.0,4.0,1,1987-1990,201.793480,50.227199,11.804575
3,17,155903859,3644,SFH,Residential,2,0.0,4.0,1,1919-1948,243.719570,50.227392,11.808091
4,18,155903856,3650,MFH,Residential,5,2.0,4.0,1,2009-,273.350410,50.231460,11.804337
5,19,155904009,3653,MFH,Residential,6,0.0,4.0,1,1991-1995,308.977787,50.227231,11.808270
6,20,155903795,3656,MFH,Residential,6,0.0,3.0,1,2009-,333.613484,50.226269,11.803069
7,21,410280928,3658,MFH,Residential,2,0.0,3.0,1,2009-,140.399313,50.226407,11.803266
8,22,155903699,3663,MFH,Residential,13,0.0,4.0,1,2009-,672.302439,50.227259,11.807772
9,23,155903731,3666,MFH,Residential,17,0.0,4.0,1,1919-1948,865.051347,50.231057,11.805092


In [67]:
# #Weather
# GRD.retrieve_weather()
# GRD.df_weather_raw.head()
# #Solar
# GRD.generate_solar()
# GRD.df_supim_solar.head()
#Electricity
GRD.generate_electricity()
GRD.df_demand_elec.head()
# # Heat
# GRD.generate_heat()
# GRD.df_demand_heat_space.head()
# # Mobility
# GRD.generate_mobility()
# GRD.df_demand_mobility.head()

,14,15,16,17,18,19,20,21,22,23,24,25,26
,electricity,electricity,electricity,electricity,electricity,electricity,electricity,electricity,electricity,electricity,electricity,electricity,electricity
0,0.777396,0.084537,0.091892,0.988684,1.208246,1.232254,1.444924,0.247869,3.157574,3.507441,3.531080,5.095485,4.120945
1,0.554303,0.127499,0.084512,0.859755,0.780094,0.775808,0.906866,0.217841,2.653432,2.870102,2.613237,4.417726,2.574791
2,0.526056,0.058634,0.066966,0.828162,0.885278,0.654489,0.646428,0.272417,2.507688,2.228743,2.120808,4.022935,3.028698
3,0.511023,0.045006,0.132911,0.561493,0.743744,0.786862,0.631744,0.255396,2.487327,1.895800,2.243385,4.023139,2.739421
4,0.526093,0.101736,0.079051,0.494204,0.752484,0.815102,0.922870,0.304681,1.495164,1.859772,2.512789,4.319668,3.088630


URBS Output Creation


In [68]:
# Weather for URBS
# GRD.create_weather_urbs()
# GRD.df_weather_urbs.head()
# # SUPIM
# GRD.create_supim()
# GRD.df_supim.head()

# Demands for URBS
GRD.create_demand()
GRD.df_demand.columns = GRD.df_demand.columns.droplevel(1)
GRD.df_demand = GRD.df_demand.reset_index(drop=True)


GRD.df_demand.head()

# # TVE
# GRD.create_tve()
# GRD.df_tve.head()

# # Bsp
# GRD.create_bsp()
# GRD.df_bsp.head()

# # Processes
# GRD.create_processes()
# GRD.df_pro.head()

# # Commodities
# GRD.create_commodities()
# GRD.df_com.head()

# # Process-Commodities mapping
# GRD.create_process_commodity()
# GRD.df_pro_com.head()

# # Storage
# GRD.create_storages()
# GRD.df_sto.head()

,14,15,16,17,18,19,20,21,22,23,24,25,26
0,0.777396,0.084537,0.091892,0.988684,1.208246,1.232254,1.444924,0.247869,3.157574,3.507441,3.531080,5.095485,4.120945
1,0.554303,0.127499,0.084512,0.859755,0.780094,0.775808,0.906866,0.217841,2.653432,2.870102,2.613237,4.417726,2.574791
2,0.526056,0.058634,0.066966,0.828162,0.885278,0.654489,0.646428,0.272417,2.507688,2.228743,2.120808,4.022935,3.028698
3,0.511023,0.045006,0.132911,0.561493,0.743744,0.786862,0.631744,0.255396,2.487327,1.895800,2.243385,4.023139,2.739421
4,0.526093,0.101736,0.079051,0.494204,0.752484,0.815102,0.922870,0.304681,1.495164,1.859772,2.512789,4.319668,3.088630


In [69]:
def replaceBusLoad(df_demand, net):
    """
    Replace column names in df_demand (which are bus indices) with corresponding load indices.
    
    Args:
        df_demand: DataFrame with bus indices as column names
        net: pandapower network containing bus and load information
    
    Returns:
        DataFrame with load indices as column names
    """
    # Create mapping from bus index to load index
    bus_to_load = {}
    for load_idx, bus_idx in net.load['bus'].items():
        bus_to_load[bus_idx] = load_idx
    
    # Get current column names (bus indices)
    current_columns = df_demand.columns.tolist()
    
    # Create new column names (load indices)
    new_columns = []
    for col in current_columns:
        if col in bus_to_load:
            new_columns.append(bus_to_load[col])
        else:
            # If bus has no load, keep original column name
            new_columns.append(col)
    
    # Rename columns
    df_demand.columns = new_columns
    
    return df_demand

In [70]:
GRD.df_demand

,14,15,16,17,18,19,20,21,22,23,24,25,26
0,0.777396,0.084537,0.091892,0.988684,1.208246,1.232254,1.444924,0.247869,3.157574,3.507441,3.531080,5.095485,4.120945
1,0.554303,0.127499,0.084512,0.859755,0.780094,0.775808,0.906866,0.217841,2.653432,2.870102,2.613237,4.417726,2.574791
2,0.526056,0.058634,0.066966,0.828162,0.885278,0.654489,0.646428,0.272417,2.507688,2.228743,2.120808,4.022935,3.028698
3,0.511023,0.045006,0.132911,0.561493,0.743744,0.786862,0.631744,0.255396,2.487327,1.895800,2.243385,4.023139,2.739421
4,0.526093,0.101736,0.079051,0.494204,0.752484,0.815102,0.922870,0.304681,1.495164,1.859772,2.512789,4.319668,3.088630
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,3.025878,0.042910,0.096010,2.186603,2.384294,3.462518,2.396430,0.244993,4.473269,8.619670,4.783636,10.268454,7.097073
8756,1.303083,0.047666,0.078760,1.494371,1.993223,2.812942,2.237266,0.242099,4.180933,6.310759,4.845813,8.632628,5.922019
8757,1.811219,0.075578,0.106621,1.365855,1.997657,1.547543,1.691810,0.287257,4.337803,7.025573,4.964762,8.389032,6.018747
8758,1.382985,0.074511,0.095281,1.318136,1.408295,1.667404,1.797044,0.241534,3.950089,5.957251,4.708424,7.097913,5.930147


In [71]:
df_demand = replaceBusLoad(GRD.df_demand,net)
df_demand.to_csv(f"/data/input/demand_{SF_paths.split('/')[-1].split('.')[0]}.csv", index=False)

In [72]:
# df = GRD.save_grid_data()